In [ ]:
# Import required libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set display options
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')

print("Libraries imported successfully")

## Step 1: Scrape JOSS Papers Data

We'll scrape the JOSS papers page to get:
- Papers currently in "REVIEW PENDING" status
- Recent papers that moved to "UNDER REVIEW"
- Submission dates and status transition dates

In [ ]:
def scrape_joss_papers(max_pages=50):
    """
    Scrape JOSS papers from XML/RSS feed pages.
    
    Returns:
        DataFrame with columns: title, status, page, submitted_at
    """
    papers = []
    base_url = "https://joss.theoj.org/papers"
    
    for page in range(1, max_pages + 1):
        url = f"{base_url}?page={page}"
        print(f"Scraping page {page}...", end=" ")
        
        try:
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            
            # Parse as XML (JOSS uses XML/RSS feeds)
            soup = BeautifulSoup(response.text, 'xml')
            
            # Find all entry tags (papers in RSS feed)
            entries = soup.find_all('entry')
            
            if not entries:
                print(f"No entries found, stopping.")
                break
            
            page_count = 0
            
            for entry in entries:
                # Get title
                title_elem = entry.find('title')
                if not title_elem:
                    continue
                
                title = title_elem.get_text(strip=True)
                
                # Get status and submission date from <content> tag
                content_elem = entry.find('content')
                status = 'UNKNOWN'
                submitted_at = None
                
                if content_elem:
                    # Get status from <state> tag
                    state_elem = content_elem.find('state')
                    if state_elem:
                        status_raw = state_elem.get_text(strip=True)
                        
                        # Map XML status to readable format
                        status_map = {
                            'review_pending': 'PRE REVIEW',
                            'under_review': 'UNDER REVIEW',
                            'published': 'PUBLISHED',
                            'rejected': 'REJECTED',
                            'withdrawn': 'WITHDRAWN'
                        }
                        
                        status = status_map.get(status_raw, status_raw.upper())
                    
                    # Get submission date from <submitted_at> tag
                    submitted_elem = content_elem.find('submitted_at')
                    if submitted_elem:
                        submitted_at = submitted_elem.get_text(strip=True)
                
                papers.append({
                    'title': title,
                    'status': status,
                    'page': page,
                    'submitted_at': submitted_at
                })
                page_count += 1
            
            print(f"found {page_count} papers")
            
            if page_count == 0:
                print("No more papers found, stopping.")
                break
                
        except Exception as e:
            print(f"Error: {e}")
            break
    
    df = pd.DataFrame(papers)
    if len(df) > 0:
        # Remove duplicates (keep first occurrence)
        df = df.drop_duplicates(subset=['title'], keep='first')
        
        # Parse submission dates
        if 'submitted_at' in df.columns:
            df['submitted_date'] = pd.to_datetime(df['submitted_at'], errors='coerce')
        
        print(f"\n✅ Scraped {len(df)} unique papers across {page} pages")
    else:
        print(f"\n⚠️ No papers found - scraping may have failed")
    
    return df

print("XML-based scraping function updated to extract status and submission date from <content> tag.")

## Step 2: Manual Data Entry (Based on Observations)

Since web scraping can be fragile, let's manually enter the data we observed:

## Debug: Examine JOSS Page Structure

Let's fetch one page and see what we're actually getting:

In [ ]:
# Fetch a sample page and test status extraction
url = "https://joss.theoj.org/papers?page=8"
print(f"Fetching: {url}\n")

try:
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'xml')
    
    # Find first entry
    entries = soup.find_all('entry')
    print(f"Found {len(entries)} entries\n")
    
    if entries:
        first_entry = entries[0]
        
        print("="*70)
        print("TESTING STATUS EXTRACTION:")
        print("="*70)
        
        # Method 1: Find content tag
        content_elem = first_entry.find('content')
        print(f"1. content_elem found: {content_elem is not None}")
        
        if content_elem:
            print(f"2. content_elem type: {type(content_elem)}")
            print(f"3. content_elem.name: {content_elem.name}")
            
            # Try to find state tag
            state_elem = content_elem.find('state')
            print(f"4. state_elem found: {state_elem is not None}")
            
            if state_elem:
                print(f"5. state_elem type: {type(state_elem)}")
                print(f"6. state_elem.name: {state_elem.name}")
                print(f"7. state_elem.string: {state_elem.string}")
                print(f"8. state_elem.get_text(): {state_elem.get_text()}")
                print(f"9. state_elem.get_text(strip=True): {state_elem.get_text(strip=True)}")
            else:
                print("\n⚠️ state_elem is None!")
                print("Examining content_elem children:")
                for child in content_elem.children:
                    if hasattr(child, 'name'):
                        print(f"  - Child: {child.name} = {child.get_text(strip=True) if child.get_text(strip=True) else 'empty'}")
        else:
            print("\n⚠️ content_elem is None!")
        
        print("\n" + "="*70)
        print("Testing with all 20 entries on this page:")
        print("="*70)
        
        status_count = {}
        for i, entry in enumerate(entries):
            title = entry.find('title').get_text(strip=True) if entry.find('title') else 'NO TITLE'
            content = entry.find('content')
            if content:
                state = content.find('state')
                if state:
                    status = state.get_text(strip=True)
                    status_count[status] = status_count.get(status, 0) + 1
                    if i < 3:  # Show first 3
                        print(f"  Paper {i+1}: {status}")
                else:
                    print(f"  Paper {i+1}: NO STATE TAG")
            else:
                print(f"  Paper {i+1}: NO CONTENT TAG")
        
        print(f"\nStatus distribution across {len(entries)} papers:")
        for status, count in status_count.items():
            print(f"  {status}: {count}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Option 1: Try automated scraping
print("Attempting to scrape JOSS papers...")
print("(This may take 1-2 minutes)\n")

use_manual_data = False
estimated_pending_ahead = 55  # Default estimate for manual mode

try:
    df_scraped = scrape_joss_papers(max_pages=40)  # Scrape first 30 pages
    
    if len(df_scraped) > 0:
        df = df_scraped
        df['is_our_paper'] = df['title'].str.contains('Molass', case=False, na=False)
        
        # Calculate days_since_submission for papers with valid dates
        if 'submitted_date' in df.columns:
            today_timestamp = pd.Timestamp('2025-12-19', tz='UTC')
            df['days_since_submission'] = (today_timestamp - df['submitted_date']).dt.days
            
            # Only keep positive values (papers submitted in the past)
            df.loc[df['days_since_submission'] < 0, 'days_since_submission'] = pd.NA
        
        print(f"\n{'='*70}")
        print("SCRAPED DATA SUMMARY")
        print('='*70)
        print(f"Total papers found: {len(df)}")
        print(f"\nStatus distribution:")
        print(df['status'].value_counts())
        
        # Show average wait time if we have the data
        if 'days_since_submission' in df.columns:
            pending_df_temp = df[df['status'].isin(['REVIEW PENDING', 'PRE REVIEW'])]
            valid_wait_times = pending_df_temp['days_since_submission'].notna().sum()
            if valid_wait_times > 0:
                avg_wait = pending_df_temp['days_since_submission'].mean()
                median_wait = pending_df_temp['days_since_submission'].median()
                print(f"\nPRE REVIEW queue wait times ({valid_wait_times} papers with valid dates):")
                print(f"  Average: {avg_wait:.1f} days")
                print(f"  Median: {median_wait:.1f} days")
                print(f"  Range: {pending_df_temp['days_since_submission'].min():.0f} - {pending_df_temp['days_since_submission'].max():.0f} days")
        
        # Find our paper
        if df['is_our_paper'].any():
            our_paper_index = df[df['is_our_paper']].index[0]
            print(f"\n✅ Found our paper: {df.loc[our_paper_index, 'title']}")
            print(f"   Overall position: {our_paper_index + 1}")
            
            # Position among REVIEW PENDING/PRE REVIEW only
            pending_df = df[df['status'].isin(['REVIEW PENDING', 'PRE REVIEW'])].reset_index(drop=True)
            if pending_df['is_our_paper'].any():
                pending_position = pending_df[pending_df['is_our_paper']].index[0] + 1
                print(f"   Position in REVIEW PENDING queue: {pending_position} out of {len(pending_df)}")
        else:
            print("\n⚠️ Our paper not found in scraped data.")
            use_manual_data = True
    else:
        print("⚠️ Scraping returned 0 papers. HTML structure may have changed.")
        use_manual_data = True
            
except Exception as e:
    print(f"\n⚠️ Scraping error: {e}")
    use_manual_data = True

# Option 2: Manual entry with CORRECTED data
print("use_manual_data", use_manual_data)
if use_manual_data:
    print("\nFalling back to manual data entry with corrected estimates...\n")
    
    # Based on: our paper is at position 169 overall (from JOSS papers page)
    # Estimate what portion are REVIEW PENDING vs UNDER REVIEW vs PUBLISHED
    
    # Reasonable estimates based on JOSS patterns:
    # - About 10-20% of active papers are REVIEW PENDING (waiting for editor)
    # - The rest are either UNDER REVIEW or recently published
    
    # Conservative estimate: assume position 169 includes many published papers
    # Let's estimate ~50-60 papers are ahead of us in REVIEW PENDING
    
    papers_data = {
        'title': ['Molass Library: A Python Package for SEC-SAXS Data Analysis'],
        'status': ['PRE REVIEW'],
        'days_since_submission': [59],  # Oct 21 to Dec 19
        'overall_position': [169],
        'is_our_paper': [True]
    }
    
    df = pd.DataFrame(papers_data)
    
    print(f"{'='*70}")
    print("MANUAL DATA WITH ESTIMATES")
    print('='*70)
    print(f"Our paper overall position (from JOSS page): 169")
    print(f"Estimated papers in REVIEW PENDING ahead of us: ~{estimated_pending_ahead}")
    print(f"Days since submission: 33")
    print(f"\n📌 Estimation logic:")
    print(f"   Position 169 includes REVIEW PENDING + UNDER REVIEW + recently PUBLISHED")
    print(f"   Assuming ~10-15% are REVIEW PENDING → ~50-60 papers ahead")
    print(f"\n⚠️  Note: These are conservative estimates based on typical JOSS patterns.")
    print(f"   Actual queue may be shorter or longer.")

print(f"\n{'='*70}")
print("Data loading complete!")
print('='*70)


## Verify Submission Order (FCFS Assumption)

Check if papers appearing before ours were actually submitted before our paper.

In [ ]:
# Verify FCFS assumption: Check if papers ahead were submitted before ours
if not use_manual_data and 'submitted_date' in df.columns:
    print("="*70)
    print("SUBMISSION DATE ANALYSIS")
    print("="*70)
    
    # Our submission date (with UTC timezone to match scraped data)
    our_submission = pd.Timestamp('2025-10-21', tz='UTC')
    
    # Filter to PRE REVIEW papers only
    pending_df = df[df['status'].isin(['REVIEW PENDING', 'PRE REVIEW'])].copy()
    
    # Check how many have valid dates
    valid_dates = pending_df['submitted_date'].notna().sum()
    print(f"\nPRE REVIEW papers with valid submission dates: {valid_dates} out of {len(pending_df)}")
    
    if valid_dates > 0:
        # Find our paper
        if 'is_our_paper' in pending_df.columns and pending_df['is_our_paper'].any():
            our_idx = pending_df[pending_df['is_our_paper']].index[0]
            papers_before_us = pending_df.loc[:our_idx-1] if our_idx > 0 else pd.DataFrame()
            
            if len(papers_before_us) > 0:
                print(f"\nPapers appearing BEFORE ours on JOSS pages: {len(papers_before_us)}")
                
                # Count by submission date
                before_ours_with_dates = papers_before_us[papers_before_us['submitted_date'].notna()]
                
                if len(before_ours_with_dates) > 0:
                    submitted_before = (before_ours_with_dates['submitted_date'] < our_submission).sum()
                    submitted_after = (before_ours_with_dates['submitted_date'] >= our_submission).sum()
                    
                    print(f"\nOf papers BEFORE us with valid dates ({len(before_ours_with_dates)} papers):")
                    print(f"  ✅ Submitted BEFORE Oct 21: {submitted_before} papers")
                    print(f"  ⚠️  Submitted AFTER Oct 21:  {submitted_after} papers")
                    
                    if submitted_after > 0:
                        print(f"\n⚠️  WARNING: {submitted_after} papers ahead of us were submitted AFTER ours!")
                        print(f"     This suggests JOSS does NOT use strict FCFS ordering.")
                        print(f"     Pages may be sorted by: paper ID, editor assignment, or other criteria.")
                        
                        # Show some examples
                        after_papers = before_ours_with_dates[before_ours_with_dates['submitted_date'] >= our_submission].sort_values('submitted_date')
                        print(f"\n     Examples of papers submitted AFTER ours but appearing before:")
                        for i, row in after_papers.head(5).iterrows():
                            days_after = (row['submitted_date'] - our_submission).days
                            print(f"       - Submitted {row['submitted_date'].strftime('%b %d')}: {days_after} days after ours")
                    else:
                        print(f"\n✅ All papers before ours were submitted before Oct 21")
                        print(f"   FCFS assumption appears valid!")
                    
                    # Check if pages are sorted by submission date
                    print(f"\n" + "-"*70)
                    print("Checking if pages are sorted by submission date:")
                    print("-"*70)
                    
                    sorted_check = pending_df[pending_df['submitted_date'].notna()].copy()
                    sorted_check['date_order'] = sorted_check['submitted_date'].rank(method='first')
                    sorted_check['page_order'] = range(len(sorted_check))
                    
                    correlation = sorted_check['date_order'].corr(sorted_check['page_order'])
                    print(f"Correlation between submission date and page order: {correlation:.3f}")
                    
                    if correlation > 0.9:
                        print("✅ Pages appear to be sorted by submission date (oldest first)")
                    elif correlation < -0.9:
                        print("⚠️  Pages appear to be sorted by submission date (newest first)")
                    else:
                        print(f"⚠️  Pages do NOT appear to be sorted by submission date")
                        print(f"   (correlation = {correlation:.3f}, expected >0.9 or <-0.9)")
        
        # Show submission date range
        print(f"\n" + "-"*70)
        print("PRE REVIEW queue submission date range:")
        print("-"*70)
        date_stats = pending_df[pending_df['submitted_date'].notna()]['submitted_date']
        if len(date_stats) > 0:
            print(f"Oldest: {date_stats.min().strftime('%B %d, %Y')}")
            print(f"Newest: {date_stats.max().strftime('%B %d, %Y')}")
            print(f"Range: {(date_stats.max() - date_stats.min()).days} days")
            print(f"\nOur submission (Oct 21) is {(our_submission - date_stats.min()).days} days after oldest")
            print(f"                          {(date_stats.max() - our_submission).days} days before newest")
    
    print("="*70)
else:
    print("\n⚠️  No submission date data available - cannot verify FCFS assumption")

## Step 3: Calculate Queue Statistics

Estimate the service rate (editor assignment rate) from observed data.

In [ ]:
# Separate papers by status (include both PRE REVIEW and REVIEW PENDING)
pending = df[df['status'].isin(['REVIEW PENDING', 'PRE REVIEW'])]
under_review = df[df['status'] == 'UNDER REVIEW'] if 'UNDER REVIEW' in df['status'].values else pd.DataFrame()

# Queue position estimation
if use_manual_data:
    # When using manual data, use the estimated value directly
    our_position = estimated_pending_ahead + 1
    papers_ahead = estimated_pending_ahead
    queue_size = estimated_pending_ahead + 1
elif 'is_our_paper' in df.columns and df['is_our_paper'].any() and 'submitted_date' in df.columns:
    # We have scraped data WITH submission dates
    our_submission = pd.Timestamp('2025-10-21', tz='UTC')
    
    # Filter to PRE REVIEW only
    pending_with_flag = pending.copy()
    if 'is_our_paper' not in pending_with_flag.columns:
        pending_with_flag['is_our_paper'] = pending_with_flag['title'].str.contains('Molass', case=False, na=False)
    
    papers_with_dates = pending_with_flag[pending_with_flag['submitted_date'].notna()]
    
    if len(papers_with_dates) > 0:
        # Count papers by submission date
        papers_before_date = (papers_with_dates['submitted_date'] < our_submission).sum()
        papers_after_date = (papers_with_dates['submitted_date'] >= our_submission).sum()
        
        # Find our position on pages (if not FCFS, this might differ)
        if pending_with_flag['is_our_paper'].any():
            page_position = pending_with_flag['is_our_paper'].tolist().index(True) + 1
        else:
            page_position = 1
        
        # Calculate submission date percentile
        our_date_rank = (papers_with_dates['submitted_date'] < our_submission).sum() + 1
        date_percentile = our_date_rank / len(papers_with_dates)
        
        print(f"📊 QUEUE POSITION ANALYSIS:")
        print(f"   Papers submitted BEFORE Oct 21: {papers_before_date}")
        print(f"   Papers submitted AFTER Oct 21: {papers_after_date}")
        print(f"   Our submission date percentile: {date_percentile:.1%} (rank {our_date_rank} of {len(papers_with_dates)})")
        print(f"   Our position on pages: {page_position} of {len(pending_with_flag)}")
        
        # Since JOSS may NOT follow strict FCFS, use a hybrid estimate
        # Use the percentile position in the queue as a more realistic estimate
        estimated_position = int(len(papers_with_dates) * date_percentile)
        papers_ahead = estimated_position - 1
        our_position = estimated_position
        queue_size = len(papers_with_dates)
        
        print(f"\n⚠️  Since JOSS does NOT strictly follow FCFS order,")
        print(f"   using percentile-based position: {our_position} of {queue_size}")
        print(f"   (This accounts for non-FCFS processing)")
    else:
        # Fallback to page order if no dates
        if pending_with_flag['is_our_paper'].any():
            our_position = pending_with_flag['is_our_paper'].tolist().index(True) + 1
        else:
            our_position = 1
        papers_ahead = our_position - 1
        queue_size = len(pending)
elif 'is_our_paper' in df.columns and df['is_our_paper'].any():
    # If we scraped successfully but no dates, use page order
    pending_with_flag = pending.copy()
    if 'is_our_paper' not in pending_with_flag.columns:
        pending_with_flag['is_our_paper'] = pending_with_flag['title'].str.contains('Molass', case=False, na=False)
    
    if pending_with_flag['is_our_paper'].any():
        our_position = pending_with_flag['is_our_paper'].tolist().index(True) + 1
    else:
        our_position = 1  # Default if not found
    
    papers_ahead = our_position - 1
    queue_size = len(pending)
else:
    # Default fallback
    our_position = 1
    papers_ahead = 0
    queue_size = len(pending)

print(f"\n{'='*70}")
print("QUEUE STATISTICS")
print('='*70)
print(f"Our estimated queue position: {our_position}")
print(f"Papers estimated ahead of us: {papers_ahead}")
print(f"Total papers in REVIEW PENDING/PRE REVIEW: {queue_size}")
print(f"Papers UNDER REVIEW (from sample): {len(under_review)}")

if len(pending) > 0 and 'days_since_submission' in pending.columns:
    print(f"\nAverage days waiting (REVIEW PENDING):")
    print(f"  Mean: {pending['days_since_submission'].mean():.1f} days")
    print(f"  Median: {pending['days_since_submission'].median():.1f} days")
    if 'is_our_paper' in pending.columns and pending['is_our_paper'].any():
        our_wait = pending[pending['is_our_paper']]['days_since_submission'].values[0]
        print(f"  Our waiting time so far: {our_wait} days")
    else:
        print(f"  Our waiting time so far: 59 days")
else:
    print(f"\nOur waiting time so far: 59 days")

print('='*70)

## Step 4: Estimate Service Rate (μ)

Estimate how many papers per day get assigned an editor (transition from REVIEW PENDING to UNDER REVIEW).

In [ ]:
# Calculate service rate (μ) from actual data if available, otherwise use estimates

if not use_manual_data and len(pending) > 0:
    # We have scraped data! Calculate service rate from it
    print("="*70)
    print("CALCULATING SERVICE RATE FROM SCRAPED DATA")
    print("="*70)
    
    # Method: Use Little's Law: L = λ * W
    # Where: L = queue length, λ = arrival rate, W = average wait time
    # Rearranging: λ = L / W (this is approximately the service rate in steady state)
    
    queue_length = len(pending)
    
    # If we have days_since_submission, use it to estimate average wait
    if 'days_since_submission' in pending.columns and pending['days_since_submission'].notna().any():
        avg_wait_days = pending['days_since_submission'].mean()
    else:
        # Estimate: assume papers wait ~60 days on average based on our observation
        avg_wait_days = 60
        print(f"⚠️  No submission dates found, assuming average wait: {avg_wait_days} days")
    
    # Calculate service rate: μ ≈ queue_length / avg_wait_days
    mu_per_day_calculated = queue_length / avg_wait_days
    mu_per_week_calculated = mu_per_day_calculated * 7
    
    print(f"\nQueue-based calculation:")
    print(f"  Current queue length: {queue_length} papers")
    print(f"  Average wait time: {avg_wait_days:.1f} days")
    print(f"  Implied service rate: {mu_per_day_calculated:.3f} papers/day")
    print(f"                       = {mu_per_week_calculated:.2f} papers/week")
    
    # Use the calculated rate for conservative estimate
    mu_per_day = mu_per_day_calculated
    mu_per_week = mu_per_week_calculated
    
    # Optimistic: 50% faster
    mu_optimistic_per_day = mu_per_day * 1.5
    
    # Pessimistic: 50% slower
    mu_pessimistic_per_day = mu_per_day * 0.5
    
    print(f"\nAdjusted estimates:")
    print(f"  Conservative (calculated): {mu_per_day:.3f} papers/day ({mu_per_week:.2f} papers/week)")
    print(f"  Optimistic (1.5x):         {mu_optimistic_per_day:.3f} papers/day ({mu_optimistic_per_day*7:.2f} papers/week)")
    print(f"  Pessimistic (0.5x):        {mu_pessimistic_per_day:.3f} papers/day ({mu_pessimistic_per_day*7:.2f} papers/week)")
    print("="*70)
    
else:
    # Fall back to manual estimates
    print("="*70)
    print("USING MANUAL SERVICE RATE ESTIMATES")
    print("="*70)
    print("(No scraped data available - using typical JOSS patterns)")
    
    # Conservative estimate: 1-2 papers per week get editor assigned
    mu_per_week = 1.5  # papers/week
    mu_per_day = mu_per_week / 7  # papers/day
    
    # Optimistic estimate: 3-4 papers per week
    mu_optimistic_per_day = 3.5 / 7
    
    # Pessimistic estimate: 0.5-1 paper per week
    mu_pessimistic_per_day = 0.75 / 7
    
    print(f"\nManual estimates based on typical JOSS behavior:")
    print(f"  Conservative: {mu_per_day:.3f} papers/day ({mu_per_week:.1f} papers/week)")
    print(f"  Optimistic:   {mu_optimistic_per_day:.3f} papers/day ({mu_optimistic_per_day*7:.1f} papers/week)")
    print(f"  Pessimistic:  {mu_pessimistic_per_day:.3f} papers/day ({mu_pessimistic_per_day*7:.1f} papers/week)")
    print("="*70)

## Step 5: Calculate Expected Waiting Time

Using queuing theory: Expected remaining wait time = (Queue position - 1) / Service rate

We subtract 1 because we're already in the queue.

In [ ]:
# Current date (automatically set to today)
today = datetime.now()
submission_date = datetime(2025, 10, 21)
days_waited = (today - submission_date).days

# Papers ahead of us in queue
papers_ahead = our_position - 1

# Expected additional waiting days
expected_wait_conservative = papers_ahead / mu_per_day
expected_wait_optimistic = papers_ahead / mu_optimistic_per_day
expected_wait_pessimistic = papers_ahead / mu_pessimistic_per_day

# Expected review start dates
expected_date_conservative = today + timedelta(days=expected_wait_conservative)
expected_date_optimistic = today + timedelta(days=expected_wait_optimistic)
expected_date_pessimistic = today + timedelta(days=expected_wait_pessimistic)

print(f"Expected Waiting Time Estimates:")
print(f"  Papers ahead of us: {papers_ahead}")
print(f"  Already waited: {days_waited} days")
print(f"\n  Conservative estimate: {expected_wait_conservative:.0f} more days")
print(f"    → Review likely to start: {expected_date_conservative.strftime('%B %d, %Y')}")
print(f"\n  Optimistic estimate: {expected_wait_optimistic:.0f} more days")
print(f"    → Review likely to start: {expected_date_optimistic.strftime('%B %d, %Y')}")
print(f"\n  Pessimistic estimate: {expected_wait_pessimistic:.0f} more days")
print(f"    → Review likely to start: {expected_date_pessimistic.strftime('%B %d, %Y')}")
print(f"\nTotal time from submission (conservative): {days_waited + expected_wait_conservative:.0f} days")

## Step 6: Confidence Intervals (Statistical)

Calculate confidence intervals assuming exponential service times (typical for M/M/c queues).

In [ ]:
# For exponential distribution, variance = 1/λ²
# Standard deviation of wait time = papers_ahead / μ²

def calculate_confidence_interval(papers_ahead, mu_per_day, confidence=0.80):
    """
    Calculate confidence interval for wait time using exponential distribution.
    
    Args:
        papers_ahead: Number of papers in front of us
        mu_per_day: Service rate (papers per day)
        confidence: Confidence level (default 80%)
    
    Returns:
        (lower_bound, expected, upper_bound) in days
    """
    expected = papers_ahead / mu_per_day
    
    # For sum of exponentials, variance = n/λ²
    variance = papers_ahead / (mu_per_day ** 2)
    std_dev = np.sqrt(variance)
    
    # Z-score for confidence interval
    z = stats.norm.ppf((1 + confidence) / 2)
    
    lower = max(0, expected - z * std_dev)
    upper = expected + z * std_dev
    
    return lower, expected, upper

# Calculate 80% confidence intervals
ci_conservative = calculate_confidence_interval(papers_ahead, mu_per_day, 0.80)
ci_optimistic = calculate_confidence_interval(papers_ahead, mu_optimistic_per_day, 0.80)

print(f"80% Confidence Intervals for Additional Wait Time:\n")
print(f"Conservative scenario:")
print(f"  {ci_conservative[0]:.0f} - {ci_conservative[2]:.0f} days")
print(f"  Expected: {ci_conservative[1]:.0f} days")
print(f"  Review start: {(today + timedelta(days=ci_conservative[0])).strftime('%b %d')} - {(today + timedelta(days=ci_conservative[2])).strftime('%b %d, %Y')}")

print(f"\nOptimistic scenario:")
print(f"  {ci_optimistic[0]:.0f} - {ci_optimistic[2]:.0f} days")
print(f"  Expected: {ci_optimistic[1]:.0f} days")
print(f"  Review start: {(today + timedelta(days=ci_optimistic[0])).strftime('%b %d')} - {(today + timedelta(days=ci_optimistic[2])).strftime('%b %d, %Y')}")

## Step 7: Visualization

In [ ]:
# Create visualization with proper sizing
fig, axes = plt.subplots(1, 2, figsize=(12, 6))  # Better aspect ratio

# Plot 1: Timeline
ax1 = axes[0]
scenarios = ['Pessimistic', 'Conservative', 'Optimistic']
dates = [expected_date_pessimistic, expected_date_conservative, expected_date_optimistic]
colors = ['#ff6b6b', '#4ecdc4', '#95e1d3']

y_pos = np.arange(len(scenarios))
days_from_now = [(d - today).days for d in dates]

bars = ax1.barh(y_pos, days_from_now, color=colors, alpha=0.7, height=0.6)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(scenarios, fontsize=11)
ax1.set_xlabel('Days from Today (Dec 19, 2025)', fontsize=11)
ax1.set_title('Expected Review Start Time\n(from today)', fontsize=13, fontweight='bold')
ax1.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Today')

# Add date labels
for i, (days, date) in enumerate(zip(days_from_now, dates)):
    ax1.text(days + 5, i, f"{days} days\n({date.strftime('%b %d')})", 
             va='center', fontsize=10, bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8))

ax1.legend(fontsize=10)
ax1.grid(axis='x', alpha=0.3)
# ax1.set_xlim(left=0)

# Plot 2: Probability distribution (conservative scenario)
ax2 = axes[1]
days_range = np.linspace(0, max(250, expected_wait_pessimistic + 50), 1000)

# Gamma distribution (sum of exponentials)
print("papers_ahead", papers_ahead)
if papers_ahead > 0:
    shape = papers_ahead
    scale = 1 / mu_per_day
    prob_density = stats.gamma.pdf(days_range, a=shape, scale=scale)
    
    ax2.plot(days_range, prob_density, 'b-', linewidth=2.5, label='Probability Density')
    ax2.fill_between(days_range, 0, prob_density, alpha=0.3, color='blue')
    ax2.axvline(x=ci_conservative[1], color='red', linestyle='--', linewidth=2.5, 
                label=f'Expected ({ci_conservative[1]:.0f} days)', zorder=5)
    ax2.axvline(x=ci_conservative[0], color='orange', linestyle=':', linewidth=2, alpha=0.8)
    ax2.axvline(x=ci_conservative[2], color='orange', linestyle=':', linewidth=2, alpha=0.8, 
                label=f'80% CI: {ci_conservative[0]:.0f}-{ci_conservative[2]:.0f} days')
else:
    ax2.text(0.5, 0.5, 'No queue - review could start anytime!', 
             ha='center', va='center', transform=ax2.transAxes, fontsize=12)

ax2.set_xlabel('Additional Days Until Review Starts', fontsize=11)
ax2.set_ylabel('Probability Density', fontsize=11)
ax2.set_title('Wait Time Distribution\n(Conservative Scenario)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10, loc='upper right')
ax2.grid(alpha=0.3)
# ax2.set_xlim(left=0)

# Adjust layout to prevent cutting off labels
plt.tight_layout()

# Save with high quality
plt.savefig('joss_review_estimate.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Visualization saved as 'joss_review_estimate.png'")
print(f"   Figure size: 12 x 6 inches (2:1 aspect ratio)")

## Summary and Recommendations

## Intuitive Explanation: Why So Long?

Let's verify the estimate makes sense with a simple analogy:


In [ ]:
print("="*70)
print("INTUITIVE EXPLANATION: IS 257 DAYS REASONABLE?")
print("="*70)

print(f"\n📊 The Math:")
print(f"   Papers ahead: {papers_ahead}")
print(f"   Service rate (conservative): {mu_per_week:.1f} papers/week")
print(f"   Expected wait: {papers_ahead} papers ÷ {mu_per_week:.1f} papers/week = {papers_ahead/mu_per_week:.1f} weeks")
print(f"                = {expected_wait_conservative:.0f} days")

print(f"\n🍔 Restaurant Queue Analogy:")
print(f"   Imagine you're at a busy restaurant with one chef.")
print(f"   - There are {papers_ahead} groups ahead of you in line")
print(f"   - The chef serves ~{mu_per_week:.1f} groups per week ({mu_per_day:.2f}/day)")
print(f"   - How long until your turn?")
print(f"     → {papers_ahead} groups ÷ {mu_per_week:.1f} groups/week = {papers_ahead/mu_per_week:.1f} weeks")

print(f"\n🔍 Why is the service rate so slow?")
print(f"   JOSS editors are volunteers with limited time:")
print(f"   - Each editor handles multiple papers simultaneously")
print(f"   - Editors must find available reviewers (can take weeks)")
print(f"   - Holiday periods slow things down (Dec/Jan)")
print(f"   - Track 2 (BCM) is one of the busiest tracks")

print(f"\n📈 Historical Evidence:")
print(f"   We've been waiting {days_waited} days already (since Oct 21)")
print(f"   At {mu_per_week:.1f} papers/week, roughly {days_waited * mu_per_day:.1f} papers")
print(f"   should have been processed in that time.")
print(f"   If the queue was originally ~{papers_ahead + days_waited * mu_per_day:.0f} papers,")
print(f"   and {days_waited * mu_per_day:.1f} moved forward, that leaves ~{papers_ahead} ahead.")

print(f"\n✅ Reality Check:")
print(f"   - The estimate assumes steady-state queue processing")
print(f"   - Could be faster if editors accelerate after holidays")
print(f"   - Could be slower if new papers keep arriving")
print(f"   - The 80% confidence interval gives a realistic range")

print(f"\n💡 Key Insight:")
print(f"   The long wait is NOT about review complexity,")
print(f"   it's about QUEUE LENGTH + LIMITED EDITOR CAPACITY.")
print(f"   Think: DMV line, not technical difficulty!")

print("="*70)

In [ ]:
print("="*70)
print("JOSS REVIEW START ESTIMATE - SUMMARY")
print("="*70)
print(f"\nPaper: Molass Library: A Python Package for SEC-SAXS Data Analysis")
print(f"Submitted: {submission_date.strftime('%B %d, %Y')}")
print(f"Today: {today.strftime('%B %d, %Y')}")
print(f"Days waited so far: {days_waited}")
print(f"\nQueue Position: {our_position} out of {queue_size} papers in REVIEW PENDING")
print(f"Papers ahead of us: {papers_ahead}")

print(f"\n" + "-"*70)
print("ESTIMATED REVIEW START DATES:")
print("-"*70)

print(f"\n📊 Conservative Estimate (most likely):")
print(f"   Expected date: {expected_date_conservative.strftime('%B %d, %Y')}")
print(f"   80% confidence: {(today + timedelta(days=ci_conservative[0])).strftime('%b %d')} - {(today + timedelta(days=ci_conservative[2])).strftime('%b %d, %Y')}")
print(f"   Additional wait: {expected_wait_conservative:.0f} days")

print(f"\n🎯 Optimistic Estimate (best case):")
print(f"   Expected date: {expected_date_optimistic.strftime('%B %d, %Y')}")
print(f"   Additional wait: {expected_wait_optimistic:.0f} days")

print(f"\n⏳ Pessimistic Estimate (worst case):")
print(f"   Expected date: {expected_date_pessimistic.strftime('%B %d, %Y')}")
print(f"   Additional wait: {expected_wait_pessimistic:.0f} days")

print(f"\n" + "="*70)
print("RECOMMENDATIONS:")
print("="*70)
print(f"\n✅ Most likely scenario: Review starts around {expected_date_conservative.strftime('%B %d, %Y')}")
print(f"\n📅 Plan for: Late January to mid-February 2026")
print(f"\n💡 Action items:")
print(f"   1. Continue monitoring JOSS review issue weekly")
print(f"   2. Prepare responses to anticipated reviewer questions")
print(f"   3. Keep molass-library repository well-maintained")
print(f"   4. Re-run this analysis monthly to update estimates")
print(f"\n⚠️  Note: These are statistical estimates. Actual timing may vary.")
print("="*70)